# 3.2 Managing Messages — Controlling What the Model Sees

## What you will learn in this notebook

- Why **message history grows unboundedly** and why that's a problem
- How **`SummarizationMiddleware`** compresses long histories into a summary
- How **`@before_agent` middleware** lets you surgically remove specific messages
- How **`RemoveMessage`** deletes individual messages by ID

---

### The context window problem

Every LLM has a **context window limit** — a maximum number of tokens it can process in one call. In long conversations, the message history grows with every turn:

```
Turn  1:   50 tokens   → fine
Turn 10:  500 tokens   → fine
Turn 50: 2500 tokens   → getting expensive
Turn 200: 10,000 tokens → context limit approached
Turn 500: 25,000 tokens → hits limit, call fails
```

Two strategies to fix this:

| Strategy | Mechanism | Best for |
|---|---|---|
| **Summarise** | Replace old messages with a short summary | General conversations where the gist matters |
| **Trim/Delete** | Remove specific message types | Removing tool noise that isn't needed later |

### What is middleware?

**Middleware** is code that runs *around* the agent loop — before or after each LLM call. It can read and modify state without changing the core agent logic. Think of it like Express/Django middleware or decorators:

```
agent.invoke()
    → middleware 1 runs (e.g. trim messages)
    → middleware 2 runs (e.g. log the request)
    → LLM call
    ← response
    ← middleware 2 post-processing
    ← middleware 1 post-processing
```

In [6]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================

from dotenv import load_dotenv

load_dotenv()

True

---
## Section 1 — Summarise Messages

**`SummarizationMiddleware`** watches the token count. When the conversation exceeds a threshold, it:
1. Takes the older messages
2. Calls a (typically cheaper) LLM to summarise them into one message
3. Replaces the old messages with that summary
4. Keeps the most recent N messages intact for full context

The result: the model always has the gist of the whole conversation, but the history stays compact.

In [2]:
# ============================================================
# CELL 2: Create an Agent With Summarisation Middleware
# ============================================================
# SummarizationMiddleware parameters:
#
#   model="gpt-4o-mini"
#     → The model used to WRITE the summary.
#       This should be a fast, cheap model — not your main model.
#       The summary doesn't need deep reasoning, just compression.
#
#   trigger=("tokens", 100)
#     → WHEN to summarise: once the history exceeds 100 tokens.
#       100 is very low — used here so summarisation triggers quickly
#       in this demo. In production, 2000-8000 tokens is typical.
#       Alternative: trigger=("messages", 20) → after 20 messages.
#
#   keep=("messages", 1)
#     → HOW MANY recent messages to keep verbatim (not summarised).
#       1 means: summarise everything except the last message.
#       In production, keep=("messages", 5) gives more recent context.
#
# The middleware= parameter accepts a list — you can stack multiple
# middleware in sequence, e.g. [summarise, log, rate_limit].

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),  # Needed to persist the summarised history
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",         # Cheap model to write the summary
            trigger=("tokens", 10000),      # Summarise when history > 100 tokens
            keep=("messages", 10),
        )
    ],
)

In [3]:
# ============================================================
# CELL 3: Invoke With a Long Pre-Built Conversation History
# ============================================================
# We inject a long conversation history directly into the invoke call.
# This simulates a conversation that has already grown long —
# normally this history would accumulate turn-by-turn via memory.
#
# The 9 messages (alternating Human/AI) easily exceed 100 tokens,
# so SummarizationMiddleware will trigger before the LLM call.
#
# What happens under the hood:
#   1. Middleware counts tokens in the message list
#   2. Detects > 100 tokens → triggers summarisation
#   3. Calls gpt-4o-mini to summarise messages [0..7]
#   4. Replaces those messages with one SystemMessage containing the summary
#   5. Keeps messages[8] (the last HumanMessage) verbatim
#   6. Main model (gpt-5-nano) now only sees: [summary_message, last_human_message]
#   7. Generates the final response

from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
    ]},
    {"configurable": {"thread_id": "1"}}
)


In [4]:
# ============================================================
# CELL 4: Inspect the Summary Message
# ============================================================
# After summarisation, response["messages"][0] is the compressed
# summary of the entire conversation history — NOT the original
# first message.
#
# The summary should mention:
#   - Lunapolis (Moon's capital)
#   - weather extremes
#   - cheese miners
#   - the union strike threat
#
# Notice how much shorter this is than the 8 original messages.
# The main model answered the final question using only this
# compact summary + the last HumanMessage.

print(response["messages"][0].content)  # The auto-generated summary

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various facts related to the fictional moon capital, Lunapolis, including its weather, population of cheese miners, and potential labor unrest.

## SUMMARY
- The capital of the moon is Lunapolis.
- The weather in Lunapolis features clear skies, with temperatures ranging from a high of 120C to a low of -100C.
- There are approximately 100,000 cheese miners residing in Lunapolis.
- The cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


---
## Section 2 — Trim / Delete Specific Messages

Sometimes you don't want to summarise — you want to **surgically remove** specific messages. Common reasons:

- **ToolMessages are bulky**: raw API responses (search results, SQL output) can be thousands of tokens. Once the agent has used the result, the raw payload is no longer needed.
- **Privacy**: remove messages containing sensitive data after they've been processed.
- **Cost control**: keep only the messages needed for the current question.

### `@before_agent` middleware

`@before_agent` decorates a function that runs **once before each agent turn** (not before each LLM call). It receives the full state and can return state updates — including `RemoveMessage` objects to delete specific messages by ID.

In [8]:
# ============================================================
# CELL 5: Define a @before_agent Middleware to Delete ToolMessages
# ============================================================
# @before_agent marks this function as pre-turn middleware.
#
# The function signature:
#   state: AgentState  → the full current state (messages, custom fields, etc.)
#   runtime: Runtime   → access to context, config, and other runtime data
#   returns: dict | None
#     → Return a dict of state updates to apply before the agent runs.
#     → Return None to make no changes.
#
# RemoveMessage(id=m.id):
#   → A special message type that tells LangGraph to DELETE the message
#     with that ID from the state.
#   → It does NOT add a message — it removes one.
#   → The ID (m.id) is the unique identifier assigned when the message
#     was created (UUID generated automatically by LangChain).
#
# Pattern used here:
#   1. Find all ToolMessages in the current state
#   2. Create a RemoveMessage for each one
#   3. Return them in the messages update list
#   4. LangGraph deletes those messages before the next LLM call
#
# This is useful when historical tool call results are bulky
# (e.g. raw Tavily JSON or large SQL result sets) and no longer
# needed for the current turn's reasoning.

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    # Find all ToolMessages (raw tool call results) in history
    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    if tool_messages:
        # Return RemoveMessage for each — LangGraph deletes them from state
        return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}
    else:
        return {"messages": messages}

In [9]:
# ============================================================
# CELL 6: Create the Agent With Trim Middleware
# ============================================================
# The trim_messages middleware will run before EVERY turn of the
# agent. So even if ToolMessages accumulated in previous turns
# (saved by InMemorySaver), they are stripped out before the LLM
# call in the next turn.

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],  # Runs before each agent turn
)

In [11]:
# ============================================================
# CELL 7: Invoke With Injected ToolMessages
# ============================================================
# We inject a conversation that includes ToolMessages (simulating
# diagnostic tool results from a previous turn).
#
# ToolMessage requires a tool_call_id — this links it to the
# AIMessage that originally requested the tool call. We use
# simple strings ("1", "2") as fake IDs for this demo.
#
# What the trim_messages middleware does:
#   Before the LLM call, it finds the two ToolMessages and removes them.
#   The model never sees "blorp-x7 initiating diagnostic ping…" or
#   "temp=42C voltage=2.9v" — those are stripped from the context.
#   The model only sees the Human/AI dialogue turns.
#
# Why this is useful:
#   The diagnostic payloads are junk tokens the model doesn't need
#   to answer "What's the temperature of the device?" — the model
#   can infer from context that it's a hardware issue.
from langchain.messages import HumanMessage, AIMessage
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),  # Will be removed
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),  # Will be removed
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
    ]},
    {"configurable": {"thread_id": "2"}}
)



NameError: name 'pprint' is not defined

In [12]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='a43414c5-6bed-40c8-aaeb-a76941cfc34a'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='3f30909f-f6e9-41e7-8c20-e79f08ea2455', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='d15a88d0-ab6e-4d62-b34f-daf1ea559f84'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='d39fba3a-b7c3-4be6-b6df-cc8638845235', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='303b47f0-e540-4e30-b646-756fecae35e7'),
              AIMessage(content='I can’t measure your device’s temperature remotely. Since it w

In [13]:
# ============================================================
# CELL 8: Print the Final Answer
# ============================================================
# The model answered based only on the Human/AI dialogue —
# the ToolMessages were stripped before the LLM received the context.
# Notice response['messages'] no longer contains ToolMessages either —
# they were deleted from state by RemoveMessage.

print(response["messages"][-1].content)

I can’t measure your device’s temperature remotely. Since it won’t turn on, you can’t see the internal temps. If the exterior feels very hot, that’s a sign of overheating. Here’s what to do:

- Power off and unplug the device. If possible, remove the battery.
- Let it cool in a well-ventilated area for at least 30 minutes (longer if it’s very hot).
- Check and clear any dust from vents using compressed air while the device is off and unplugged.
- Inspect the charger and cable for damage; try using a different compatible charger if you have one.
- After cooling, try powering on with minimal peripherals connected.

If it still won’t turn on after cooling, or it gets hot again quickly once you try to power it, there could be a hardware issue (battery, power IC, motherboard). In that case, contact the manufacturer’s support or take it to a service center.

If you want, tell me the exact device (brand and model) and what you’ve observed (any lights, sounds, screen behavior). I can tailor st

---
## Summary

| Concept | Key takeaway |
|---|---|
| **Context window problem** | Long histories consume tokens, increase cost, and eventually hit model limits |
| **`SummarizationMiddleware`** | Compresses old messages into a summary when token count exceeds a threshold |
| **`trigger=("tokens", N)`** | Summarise when history > N tokens; also `("messages", N)` for message count |
| **`keep=("messages", N)`** | How many recent messages to keep verbatim after summarisation |
| **`@before_agent`** | Decorator for a function that runs once before each agent turn |
| **`RemoveMessage(id=...)`** | Deletes a specific message from state by its ID |
| **Middleware stacking** | Pass a list to `middleware=[]` — they run in order |

### When to use which strategy

```
Long general conversation  →  SummarizationMiddleware
Bulky tool call results    →  @before_agent + RemoveMessage
Both                       →  Stack both in middleware=[...]
```